# Kline Quant · Backtrader 云端示例
在 Google Colab 中运行官方 Backtrader，上传自己的 CSV（Date,Open,High,Low,Close,Volume）。

**风险提示：** 示例仅用于教学。需要自己核对复权、交易日、滑点、手续费和时间戳；不保证实际可交易性。


In [ ]:
!pip -q install backtrader pandas matplotlib


上传证券日线CSV。多资产表若包含 Ticker 列，会优先使用所选第一只证券；若要多资产请自行建立多数据流。


In [ ]:
from google.colab import files
import io, pandas as pd, backtrader as bt
uploaded = files.upload()
filename = next(iter(uploaded))
df = pd.read_csv(io.BytesIO(uploaded[filename]))
df.columns = [c.strip().lower() for c in df.columns]
if 'ticker' in df.columns:
    ticker = sorted(df['ticker'].dropna().unique())[0]
    print('Selected ticker:', ticker)
    df = df[df['ticker'] == ticker].copy()
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date').drop_duplicates('date').set_index('date')
for column in ['open','close']:
    if column not in df.columns:
        raise ValueError(f'Missing required column {column}')
for column in ['high','low']:
    if column not in df.columns: df[column] = df['close']
if 'volume' not in df.columns: df['volume'] = 0
df = df[['open','high','low','close','volume']].dropna()
print(df.shape, df.index.min(), df.index.max())


In [ ]:
class SMACross(bt.Strategy):
    params = dict(fast=10, slow=40)
    def __init__(self):
        fast = bt.ind.SMA(self.data.close, period=self.p.fast)
        slow = bt.ind.SMA(self.data.close, period=self.p.slow)
        self.cross = bt.ind.CrossOver(fast, slow)
    def next(self):
        if not self.position and self.cross > 0:
            self.buy()
        elif self.position and self.cross < 0:
            self.close()

cerebro = bt.Cerebro()
cerebro.adddata(bt.feeds.PandasData(dataname=df))
cerebro.addstrategy(SMACross)
cerebro.broker.setcash(100000.0)
cerebro.broker.setcommission(commission=0.0003)
cerebro.addsizer(bt.sizers.PercentSizer, percents=90)
cerebro.addanalyzer(bt.analyzers.SharpeRatio_A, _name='sharpe')
cerebro.addanalyzer(bt.analyzers.DrawDown, _name='drawdown')
cerebro.addanalyzer(bt.analyzers.TradeAnalyzer, _name='trades')
cerebro.addanalyzer(bt.analyzers.Returns, _name='returns')
result = cerebro.run()[0]
print('Final equity:', round(cerebro.broker.getvalue(),2))
print('Annual Sharpe:', result.analyzers.sharpe.get_analysis())
print('Max drawdown:', result.analyzers.drawdown.get_analysis())
print('Trades:', result.analyzers.trades.get_analysis())
print('Returns:', result.analyzers.returns.get_analysis())


完整 Backtrader 引擎与站内轻量回测引擎在数据、订单撮合、再平衡和报告公式上可能不同。不能跨平台直接视为同一回测结果。
